# Hands-On Lab: Understanding Localhost and Ports (VS Code on Windows)

This notebook is designed to run inside **VS Code on Windows** using the Jupyter extension.
All code cells include **inline comments** explaining what each command does and why it is needed.


## Part 1: Create a working directory and file
We create a folder and a simple text file that will later be served by a local web server.


In [ ]:
from pathlib import Path  # Used for safe, cross-platform file handling

# Define a working directory relative to the notebook location
workdir = Path.cwd() / "localhost_lab"

# Create the directory if it does not already exist
workdir.mkdir(exist_ok=True)

# Create a text file that the web server will expose
hello_file = workdir / "hello.txt"
hello_file.write_text(
    "Hello from localhost on port 8000 (Windows / VS Code)\n",
    encoding="utf-8"
)

# Display what was created
print("Created file:", hello_file)
print("Directory contents:", [p.name for p in workdir.iterdir()])


## Part 2: Start a local web service on port 8000
We start a simple HTTP server using Python's built-in module.


In [ ]:
import subprocess, time, sys

# Launch a Python HTTP server as a separate process
# - sys.executable ensures the same Python environment is used
# - http.server starts a simple local web server
# - 8000 is the port number to listen on
server_proc_8000 = subprocess.Popen(
    [sys.executable, "-m", "http.server", "8000"],
    cwd=str(workdir),           # Serve files from the working directory
    stdout=subprocess.PIPE,     # Capture server logs
    stderr=subprocess.STDOUT,
    text=True
)

# Wait briefly so the server can bind to the port
time.sleep(1)

print("Server started on port 8000.")
print("Open in browser: http://localhost:8000")
print("Open in browser: http://127.0.0.1:8000")

# Display initial server output
for _ in range(2):
    line = server_proc_8000.stdout.readline().strip()
    if line:
        print("SERVER LOG:", line)


## Part 3: Confirm the service responds
We request data from the server using Python.


In [ ]:
import urllib.request

# Send an HTTP request to the local server
with urllib.request.urlopen("http://localhost:8000") as response:
    html = response.read(500).decode("utf-8", errors="replace")

print("Response received from localhost:8000")
print(html)


## Part 4: Fetch a specific file from the server
This confirms that files are served correctly through the port.


In [ ]:
with urllib.request.urlopen("http://localhost:8000/hello.txt") as response:
    text = response.read().decode("utf-8", errors="replace")

print("File content served over HTTP:")
print(text)


## Part 5: Try the wrong port
Connecting to a port with no service should fail.


In [ ]:
import urllib.error

try:
    urllib.request.urlopen("http://localhost:9000", timeout=2)
    print("Unexpected success")
except Exception as e:
    print("Expected failure.")
    print("No service is listening on port 9000.")
    print("Error:", type(e).__name__)


## Part 6: Start a second service on port 9000
Both services run on the same machine using different ports.


In [ ]:
server_proc_9000 = subprocess.Popen(
    [sys.executable, "-m", "http.server", "9000"],
    cwd=str(workdir),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(1)

print("Second server started on port 9000.")
print("Open in browser: http://localhost:9000")


## Part 7: Verify both services
Each port responds independently.


In [ ]:
for port in (8000, 9000):
    url = f"http://localhost:{port}/hello.txt"
    with urllib.request.urlopen(url) as response:
        text = response.read().decode("utf-8", errors="replace").strip()
    print(f"Response from port {port}: {text}")


## Part 8: Inspect listening ports on Windows
We use netstat to confirm which ports are active.


In [ ]:
import subprocess, platform

if platform.system().lower() == "windows":
    out_8000 = subprocess.check_output(
        r'netstat -ano | findstr ":8000"',
        shell=True,
        text=True,
        errors="replace"
    )
    out_9000 = subprocess.check_output(
        r'netstat -ano | findstr ":9000"',
        shell=True,
        text=True,
        errors="replace"
    )
    print("Port 8000 entries:")
    print(out_8000.strip() or "(none)")
    print("\nPort 9000 entries:")
    print(out_9000.strip() or "(none)")
else:
    print("This section is designed for Windows.")


## Part 9: Clean up
Stop the servers to free the ports.


In [ ]:
def stop_process(proc, name):
    if proc and proc.poll() is None:
        proc.terminate()   # Ask process to stop gracefully
        try:
            proc.wait(timeout=3)
        except Exception:
            proc.kill()    # Force stop if needed
        print(f"Stopped {name}.")
    else:
        print(f"{name} not running.")

stop_process(server_proc_8000, "server on port 8000")
stop_process(server_proc_9000, "server on port 9000")


## Key takeaways
- Localhost always means your own machine
- Ports identify services on that machine
- Multiple services can run simultaneously using different ports
- The same concepts power Jupyter, Streamlit, APIs, and SSH
